# DepthFM (CompVis)

Relative depth via flow matching. Downloads a ~1.7GB checkpoint.

**Runtime:** GPU (L4 or A100)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, time, gc, sys, json, importlib, subprocess
import numpy as np
import cv2
import torch
from PIL import Image

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

assert os.path.isdir(DATASET_IMAGES), f'Dataset not found: {DATASET_IMAGES}'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_depth(depth_np, stem, model_name):
    d = np.array(depth_np, dtype=np.float32)
    while d.ndim > 2: d = d[0]
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    for sub, img in [('depth', (d_norm*65535).astype(np.uint16)),
                     ('vis',   cv2.applyColorMap((d_norm*255).astype(np.uint8), cv2.COLORMAP_INFERNO))]:
        p = os.path.join(SAVE_DIR, model_name, sub)
        os.makedirs(p, exist_ok=True)
        cv2.imwrite(os.path.join(p, f'{stem}.png'), img)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Setup done.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Load shared sample list from Drive (run depth_pro.ipynb first)
assert os.path.exists(SAMPLE_FILE), f'Run depth_pro.ipynb first to create sample_images.txt'
SAMPLES = [l.strip() for l in open(SAMPLE_FILE) if l.strip()]
print(f'Loaded {len(SAMPLES)} samples')

## Install + download checkpoint

In [ ]:
!git clone https://github.com/CompVis/depth-fm /content/depth-fm 2>/dev/null || true
# Install torchdiffeq FIRST (grep -v '^torch' would accidentally strip it)
!pip install -q torchdiffeq einops omegaconf
!grep -v '^torch==' /content/depth-fm/requirements.txt | pip install -q -r /dev/stdin
!mkdir -p /content/depth-fm/checkpoints
![ -f /content/depth-fm/checkpoints/depthfm-v1.ckpt ] || \
    wget -q --show-progress -O /content/depth-fm/checkpoints/depthfm-v1.ckpt \
    https://ommer-lab.com/files/depthfm/depthfm-v1.ckpt

## Run

In [ ]:
sys.path.insert(0, '/content/depth-fm')
importlib.invalidate_caches()
from depthfm import DepthFM

MODEL_NAME = 'depthfm'
model = DepthFM('/content/depth-fm/checkpoints/depthfm-v1.ckpt')
model.cuda().eval()

times = []
for img_name in SAMPLES:
    stem = img_name.split('.')[0]
    img = Image.open(os.path.join(DATASET_IMAGES, img_name)).convert('RGB')
    w, h = img.size
    img = img.resize(((w//64)*64 or 64, (h//64)*64 or 64), Image.LANCZOS)
    t = torch.tensor(np.array(img).astype(np.float32)).permute(2,0,1).unsqueeze(0)/127.5-1.0
    t0 = time.time()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.float16):
            depth_t = model.predict_depth(t.cuda(), num_steps=2, ensemble_size=1)
    torch.cuda.synchronize(); times.append(time.time()-t0)
    save_depth(depth_t[0,0].cpu().numpy(), stem, MODEL_NAME)

print(f'Done -- {len(times)} images, avg {np.mean(times):.3f}s/img')
del model; sys.path.remove('/content/depth-fm'); clear_gpu()

## Confirm saved

In [ ]:
model_dir = os.path.join(SAVE_DIR, MODEL_NAME)
n_depth = len(os.listdir(os.path.join(model_dir, 'depth')))
n_vis   = len(os.listdir(os.path.join(model_dir, 'vis')))
print(f'{MODEL_NAME}: {n_depth} depth maps, {n_vis} visualizations saved to Drive')